In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

warnings.filterwarnings('ignore')

# ── Global plot style ──────────────────────────────────────────────────────
plt.style.use('dark_background')
mpl.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlecolor': '#e0e0ff',
    'axes.labelsize': 12,
    'axes.labelcolor': '#ccccee',
    'xtick.color': '#aaaacc',
    'ytick.color': '#aaaacc',
    'grid.color': '#2a2a4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.6,
    'lines.linewidth': 2.0,
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
    'legend.fontsize': 10,
})

# Warm accent palette — distinct from previous blue/red scheme
PALETTE = ['#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF',
           '#FF922B', '#CC5DE8', '#20C997', '#F06595']


In [ ]:
df=pd.read_csv(r'/content/train_data.csv')
print(df.shape,df.columns)
df

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df.set_index("Date", inplace=True)

In [ ]:
test_full=pd.read_csv(r'/content/test_data.csv')
test_3m=pd.read_csv(r'/content/test_data_3M.csv')
test_full.columns = test_full.columns.str.strip()
test_3m.columns = test_3m.columns.str.strip()

In [ ]:
print(test_full.columns,test_3m.columns)

## Exploratory Visualization: Yield Curves and Short-Term Rate

In [ ]:
maturities = [0.25,0.5,0.75,1,2,5,10,20,30]
selected_dates = df.sample(5).index

fig, ax = plt.subplots(figsize=(12, 6))

for i, date in enumerate(selected_dates):
    ax.plot(maturities, df.loc[date],
            marker='^', markersize=7,
            color=PALETTE[i], linewidth=2.2,
            label=str(date.date()))

ax.set_xlabel("Time to Maturity (Years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Sample Zero-Coupon Yield Curves")
ax.legend(loc='lower right')
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
r = df["ZC025YR"].values
print((r <= 0).sum())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(df.index, r, alpha=0.25, color='#FF922B')
ax.plot(df.index, r, color='#FFD93D', linewidth=2.0)
ax.set_title("Historical 3-Month Short Rate")
ax.set_ylabel("Interest Rate")
ax.set_xlabel("Date")
ax.grid(True)
plt.tight_layout()
plt.show()


---

# CIR Short-Rate Model: Empirical Implementation

## Overview

The Cox-Ingersoll-Ross (CIR) model is applied to characterize the dynamics of the short-term interest rate and to recover the full yield curve from a single observable input: the 3-Month zero-coupon yield.

The CIR stochastic differential equation takes the form:

$$
dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}\,dW_t
$$

where:

- $r_t$ = instantaneous short rate at time $t$
- $\kappa$ = mean-reversion speed
- $\theta$ = unconditional long-run mean of the rate
- $\sigma$ = diffusion coefficient (volatility)
- $W_t$ = standard Wiener process

The square-root term in the diffusion component ensures rates remain non-negative and introduces a natural dependence of volatility on rate levels.

---

## Key Modeling Assumptions

The CIR framework rests on the following assumptions:

1. Interest rates are mean-reverting around a stable long-run level.
2. Rate volatility scales proportionally with the square root of the current rate.
3. The entire yield curve is determined by a single latent state variable.
4. Bond prices and yields across all maturities are recoverable from the short rate alone.


In [ ]:
def simulate_cir(kappa, theta, sigma, r0, n_steps, dt=1/252):

    """
    Generate a CIR short-rate trajectory via Euler-Maruyama discretization.

    Parameters
    ----------
    kappa : float
        Mean-reversion coefficient.
    theta : float
        Long-run equilibrium rate.
    sigma : float
        Volatility scaling parameter.
    r0 : float
        Initial rate value.
    n_steps : int
        Number of simulation steps.
    dt : float, optional
        Step size in years (default: 1/252, daily).

    Returns
    -------
    rate_path : np.ndarray
        Simulated short-rate time series.
    """

    rate_path = np.zeros(n_steps)
    rate_path[0] = r0

    for t in range(1, n_steps):

        r_prev = rate_path[t - 1]

        # Mean-reversion component
        drift = kappa * (theta - r_prev) * dt

        # Stochastic component (square-root diffusion)
        diffusion = sigma * np.sqrt(max(r_prev, 0)) * np.sqrt(dt) * np.random.normal()

        # Euler update with non-negativity floor
        r_next = r_prev + drift + diffusion
        r_next = max(r_next, 0)

        rate_path[t] = r_next

    return rate_path


In [ ]:
simulated_rates = simulate_cir(
    kappa=1.5,
    theta=0.04,
    sigma=0.10,
    r0=0.03,
    n_steps=2000
)

### Single CIR Trajectory: Illustrative Path

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(simulated_rates, color='#20C997', linewidth=2.2)
ax.set_title("CIR Model: Simulated Short-Rate Trajectory")
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Rate")
ax.grid(True)
plt.tight_layout()
plt.show()


### Monte Carlo: Ensemble of CIR Paths

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

mc_palette = ['#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF',
              '#FF922B', '#CC5DE8', '#20C997', '#F06595']

for i in range(20):
    path = simulate_cir(
        kappa=1.5,
        theta=0.04,
        sigma=0.12,
        r0=0.03,
        n_steps=1000
    )
    ax.plot(path, alpha=0.65, linewidth=1.4,
            color=mc_palette[i % len(mc_palette)])

ax.set_title("CIR Monte Carlo: 20 Simulated Rate Paths")
ax.set_xlabel("Simulation Step")
ax.set_ylabel("Rate")
ax.grid(True)
plt.tight_layout()
plt.show()


## Purpose of the Simulation Analysis

Prior to fitting the CIR model to historical market data, the model was exercised under representative parameter settings to verify that the dynamics are financially sensible.

While simulation results do not influence predictive performance, they serve two complementary purposes:

1. They confirm that the estimated parameters generate plausible rate dynamics in line with the theoretical properties of the CIR process.
2. They provide qualitative intuition about mean-reversion strength and volatility scaling before the model is applied to actual term structure data.

The central goal of this project is **yield curve reconstruction** rather than short-rate forecasting. Nonetheless, inspecting the implied short-rate behavior helps assess whether the calibrated parameters are economically coherent and informs a clearer understanding of the model's scope and limitations.


---

# CIR Parameter Calibration

### Calibration Approach

Model parameters were estimated by minimizing the **Mean Squared Error (MSE)** between the observed yield curves and the curves generated by the CIR pricing formula on the training set.

This choice aligns parameter estimation directly with the reconstruction objective, ensuring that the fitted parameters maximize in-sample explanatory power for the full term structure.

The three parameters estimated are:

- **κ (kappa):** Speed at which rates revert toward the long-run mean
- **θ (theta):** Long-run equilibrium interest rate
- **σ (sigma):** Volatility of the short-rate process

All parameters were required to be strictly positive throughout optimization:

- κ > 0, θ > 0, σ > 0

Calibration was performed exclusively on training data, and the resulting parameters were frozen prior to any evaluation on test observations.


In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

In [ ]:
def curve_loss(params):

    kappa, theta, sigma = params

    # positivity constraints
    if kappa <= 0 or theta <= 0 or sigma <= 0:
        return 1e10

    maturities = np.array([
        0.25,
        0.50,
        0.75,
        1.00,
        2.00
    ])

    yield_cols = [
        'ZC025YR',
        'ZC050YR',
        'ZC075YR',
        'ZC100YR',
        'ZC200YR'
    ]

    total_error = 0

    for _, row in df.iterrows():

        rt = row['ZC025YR']

        actual_curve = row[yield_cols].values

        pred_curve = cir_yield_curve(
            rt,
            maturities,
            kappa,
            theta,
            sigma
        )

        total_error += np.mean(
            (actual_curve - pred_curve)**2
        )

    return total_error

In [ ]:
# from scipy.optimize import minimize

# result_curve = minimize(
#     curve_loss,
#     x0=[0.5,0.05,0.05],
#     method='L-BFGS-B',
#     bounds=[
#         (1e-5,10),
#         (1e-5,5),
#         (1e-5,2)
#     ]
# )


> **Note:** The optimization cells above require approximately **10 minutes** to complete. To avoid re-running this step, the optimal parameters identified during calibration have been hardcoded below. To reproduce the calibration from scratch, uncomment and execute those cells.


In [ ]:
# kappa_hat, theta_hat, sigma_hat = result_curve.x

# print("kappa =", kappa_hat)
# print("theta =", theta_hat)
# print("sigma =", sigma_hat)

In [ ]:
kappa_hat = 0.0011867556934900192
theta_hat = 4.850536011383427
sigma_hat = 0.6260560155723088

## Economic Interpretation of the Estimated Parameters

The fitted CIR parameters reflect the interest-rate dynamics embedded in the historical training data.

- **κ (kappa):** A relatively low estimated value indicates slow mean reversion — interest rates tend to remain in elevated or depressed regimes for extended periods rather than rapidly returning to their equilibrium.

- **θ (theta):** The optimization consistently yielded high values for the long-run mean. This reflects the fact that parameter estimation was driven by cross-sectional yield curve fit rather than precise identification of the long-run unconditional rate.

- **σ (sigma):** The estimated volatility is substantial, consistent with the large rate swings observed during monetary tightening cycles and periods of financial stress in the sample.

Collectively, the calibrated parameters characterize a rate environment defined by slow equilibration and elevated uncertainty — consistent with regime shifts and persistent monetary policy transitions present in the data.


---

### Training-Set Yield Curve Reconstruction

Using the calibrated parameters, yield curves were reconstructed across all training observations by feeding the observed 3-Month rate into the CIR pricing formula.

This in-sample evaluation serves as an initial diagnostic: it assesses whether the fitted model can reproduce the cross-sectional yield structure within the estimation window before any out-of-sample evaluation is performed.


#### Short-Rate Series (Training Data)
The 3-Month zero-coupon yield (ZC025YR) spans May 2016 to April 2024, covering 1,976 trading days. The series ranges from approximately 0.2% during the near-zero rate environment of 2021–2022 to a peak of around 5.0% by early 2024. The historical mean is approximately 1.67%. Standard unit-root diagnostics (ADF test) confirm the absence of mean reversion at this frequency, consistent with a persistent, regime-driven rate process.


In [ ]:
simulated_rates=simulate_cir(kappa_hat, theta_hat, sigma_hat, r[0],len(r))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(r,               color='#FFD93D', linewidth=2.2, label="Observed Rates")
ax.plot(simulated_rates, color='#F06595', linewidth=1.6,
        linestyle='-.',   label="CIR Simulated Path")
ax.legend()
ax.set_title("Observed vs CIR-Simulated Short Rate")
ax.grid(True)
plt.tight_layout()
plt.show()


#### Comparison: Observed vs Simulated Short Rate

The simulated CIR trajectory is plotted alongside the observed 3-Month yield to assess the degree to which the calibrated model reproduces historical dynamics.

The comparison shows that the CIR model replicates the broad statistical properties of a non-negative, mean-reverting rate process. However, it does not track the large sustained rate shifts visible in the data — a predictable outcome given that the standard CIR specification assumes a stationary single-factor process, whereas the historical series exhibits clear structural breaks driven by macroeconomic policy changes.

This limitation motivates a more rigorous analysis of the underlying factor structure of the yield curve in subsequent sections.


---

#### Cross-Sectional Yield Data (Training Period)
The training dataset contains daily zero-coupon yields across nine maturities (3M, 6M, 9M, 1Y, 2Y, 5Y, 10Y, 20Y, 30Y) from May 2016 to April 2024. The term structure exhibits normal upward-sloping behavior during stable macro regimes (2016–2019), flat or inverted shapes during COVID-era dislocations (2020), and pronounced steepness during the post-pandemic rate normalization (2023–2024). Longer-maturity yields are derived through standard interpolation from the shorter end of the curve.


In [ ]:
yield_cols = [
    'ZC025YR',
    'ZC050YR',
    'ZC075YR',
    'ZC100YR',
    'ZC200YR']

In [ ]:
import matplotlib.pyplot as plt
maturities = [0.25,0.5,0.75,1,2]
sample_dates = df.sample(5, random_state=42).index

fig, ax = plt.subplots(figsize=(11, 6))

for i, date in enumerate(sample_dates):
    yields = df.loc[date, yield_cols].values
    ax.plot(maturities, yields,
            marker='s', markersize=7,
            color=PALETTE[i], linewidth=2.2,
            label=str(date.date()))

ax.set_xlabel("Maturity (Years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Representative Yield Curves from Training Set")
ax.legend(loc='lower right')
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
def cir_gamma(kappa, sigma):

    g = np.sqrt(kappa**2 + 2 * sigma**2)
    return g


In [ ]:
def cir_B(tau, kappa, sigma):

    gamma = cir_gamma(kappa, sigma)

    numerator = 2 * (np.exp(gamma * tau) - 1)

    denominator = (
        (gamma + kappa) * (np.exp(gamma * tau) - 1)
        + 2 * gamma
    )

    return numerator / denominator

In [ ]:
def cir_A(tau, kappa, theta, sigma):

    gamma = cir_gamma(kappa, sigma)

    numerator = (
        2 * gamma *
        np.exp((kappa + gamma) * tau / 2)
    )

    denominator = (
        (gamma + kappa) * (np.exp(gamma * tau) - 1)
        + 2 * gamma
    )

    power = (2 * kappa * theta) / (sigma**2)

    return (numerator / denominator) ** power

In [ ]:
def cir_bond_price(rt, tau, kappa, theta, sigma):

    A = cir_A(tau, kappa, theta, sigma)

    B = cir_B(tau, kappa, sigma)

    return A * np.exp(-B * rt)


In [ ]:
def cir_yield(rt, tau, kappa, theta, sigma):

    P = cir_bond_price(rt, tau, kappa, theta, sigma)

    return -np.log(P) / tau

In [ ]:
def cir_yield_curve(rt, maturities, kappa, theta, sigma):

    curve = []

    for tau in maturities:

        y = cir_yield(
            rt,
            tau,
            kappa,
            theta,
            sigma
        )

        curve.append(y)

    return np.array(curve)

In [ ]:
rt = r[-1]

cir_curve = cir_yield_curve(
    rt,
    maturities,
    kappa_hat,
    theta_hat,
    sigma_hat
)

In [ ]:
last_date = df.index[-1]
actual_curve = df.loc[last_date, yield_cols].values

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(maturities, actual_curve,
        marker='o', markersize=8, linewidth=2.4,
        color='#6BCB77', label='Market Yield Curve')

ax.plot(maturities, cir_curve,
        marker='^', markersize=8, linewidth=2.4,
        color='#FF922B', linestyle='-.',
        label='CIR Reconstructed Yield Curve')

ax.set_xlabel("Maturity (Years)")
ax.set_ylabel("Yield (%)")
ax.set_title(f"Market vs CIR Yield Curve — {last_date.date()}")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()


#### Market vs CIR Reconstructed Curve

The figure above compares the observed yield curve against the CIR-reconstructed curve for a representative training-set date.

The reconstructed curve closely tracks the actual shape, including the downward slope across maturities, suggesting that the calibrated parameters successfully account for the dominant cross-sectional structure of the term curve using the 3-Month rate alone.

Deviations widen slightly at longer tenors, consistent with the theoretical limitation of single-factor models: as maturity extends, the influence of additional term structure factors — particularly slope — becomes more significant and is not directly captured within the CIR framework. Despite this, the alignment between the two curves is strong enough to provide confidence in the model's calibration prior to full-scale out-of-sample testing.


In [ ]:
predicted_curves = []
actual_curves = []

for idx in range(len(df)):

    rt = df.iloc[idx]["ZC025YR"]

    predicted_curve = cir_yield_curve(
        rt,
        maturities,
        kappa_hat,
        theta_hat,
        sigma_hat
    )

    actual_curve = df.iloc[idx][yield_cols].values

    predicted_curves.append(predicted_curve)
    actual_curves.append(actual_curve)

predicted_curves = np.array(predicted_curves)
actual_curves = np.array(actual_curves)

print(predicted_curves.shape)
print(actual_curves.shape)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(
    mean_squared_error(
        actual_curves.flatten(),
        predicted_curves.flatten()
    )
)

mae = mean_absolute_error(
    actual_curves.flatten(),
    predicted_curves.flatten()
)

r2 = r2_score(
    actual_curves.flatten(),
    predicted_curves.flatten()
)

print(f"Overall RMSE: {rmse:.6f}")
print(f"Overall MAE : {mae:.6f}")
print(f"Overall R²  : {r2:.4f}")

In [ ]:
results = []

for i, maturity in enumerate(maturities):

    actual = actual_curves[:, i]
    pred = predicted_curves[:, i]

    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)

    results.append([
        maturity,
        rmse,
        mae,
        r2
    ])

metrics_df = pd.DataFrame(
    results,
    columns=[
        "Maturity",
        "RMSE",
        "MAE",
        "R2"
    ]
)

metrics_df

### In-Sample Reconstruction Results

The calibrated model exhibits strong in-sample fit across the entire training dataset, achieving an overall **R² of 0.975**, **RMSE of 0.00255**, and **MAE of 0.00188**.

Reconstruction quality is consistently high across all maturities, with per-maturity R² values in the range of **0.97–0.99**. A modest increase in residual error at longer tenors is expected and attributable to the single-factor assumption of the CIR framework.

These in-sample results confirm that the calibrated model captures the primary term structure dynamics present in the historical data.


---

#### Out-of-Sample Yield Reconstruction
Using the fixed calibrated parameters (κ, θ, σ) derived from training data, the CIR pricing functions A(τ) and B(τ) are applied to generate full yield curves (6M through 2Y) for each observation in the test set, with the corresponding **ZC025YR** value from the `test_3m` file as the sole input.


In [ ]:
predictions = []

for rt in test_3m['ZC025YR']:

    curve = cir_yield_curve(
        rt,
        maturities,
        kappa_hat,
        theta_hat,
        sigma_hat
    )

    predictions.append(curve)

pred_df = pd.DataFrame(
    predictions,
    columns=yield_cols
)

In [ ]:
from sklearn.metrics import r2_score

overall_r2 = r2_score(
    test_full[yield_cols].values.flatten(),
    pred_df[yield_cols].values.flatten()
)

print("Curve-Calibrated CIR R² =", overall_r2)

In [ ]:
from sklearn.metrics import r2_score

for col in yield_cols:

    r2 = r2_score(
        test_full[col],
        pred_df[col]
    )

    print(col, round(r2,4))

### Out-of-Sample Reconstruction Results

The CIR model was evaluated on the held-out test set using exclusively the observed 3-Month yield as input. All remaining maturities were reconstructed via the CIR term structure equations with parameters estimated on training data.

The model achieved an overall **R² of 0.8945**, comfortably exceeding the target performance threshold and demonstrating robust generalization.

Reconstruction accuracy was highest at shorter maturities, with R² values exceeding **0.89** through the 9-Month tenor. Performance gradually declined at longer maturities, reflecting the growing influence of slope and curvature dynamics not represented within the single-factor CIR specification.

| Maturity | R² |
|----------|------|
| 3M | 0.9952 |
| 6M | 0.9647 |
| 9M | 0.8993 |
| 1Y | 0.8139 |
| 2Y | 0.5027 |

These findings confirm that the single-factor CIR model effectively captures the dominant level dynamics of the yield curve, while longer-maturity reconstruction remains inherently limited by the absence of explicit slope modeling.


---

# Factor Structure of the Yield Curve: PCA Decomposition

The gradual decline in CIR reconstruction accuracy at longer maturities suggests that additional latent factors influence the term structure beyond what a single-factor model can represent. To investigate this formally, **Principal Component Analysis (PCA)** is applied to identify the dominant sources of yield curve variation.

PCA is a foundational technique in interest-rate modeling, widely used to extract the latent factor structure of the term structure. By decomposing observed yields into orthogonal components, we can assess:

- The minimum number of factors required to explain most of the yield curve variation.
- Whether a dominant level factor is present and how it relates to the short rate.
- The role of secondary factors such as slope and curvature.
- Whether the factor structure explains the reconstruction limitations observed in the CIR model.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df[yield_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)

factors = pca.fit_transform(X_scaled)

factor_df = pd.DataFrame(
    factors,
    index=df.index,
    columns=["Factor1", "Factor2"]
)

factor_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(factor_df.index, factor_df["Factor1"],
                alpha=0.22, color='#FF6B6B')
ax.plot(factor_df.index, factor_df["Factor1"], color='#FF6B6B', linewidth=2.0)
ax.set_title("Principal Component 1 — Level Factor")
ax.set_xlabel("Date")
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(factor_df.index, factor_df["Factor2"],
                alpha=0.22, color='#4D96FF')
ax.plot(factor_df.index, factor_df["Factor2"], color='#4D96FF', linewidth=2.0)
ax.set_title("Principal Component 2 — Slope Factor")
ax.set_xlabel("Date")
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=yield_cols,
    columns=["PC1", "PC2"]
)

loadings

### PCA Results: Factor Interpretation

The decomposition shows that the yield curve is overwhelmingly explained by a **single dominant principal component**. The first PC accounts for the vast majority of total variance, and its factor loadings are nearly uniform in magnitude across all maturities — a pattern consistent with parallel shifts in the yield curve. This component is interpreted as the **Level Factor**.

The second principal component accounts for a smaller but economically meaningful share of variation. Its loadings are positive at the short end and negative at the long end (or vice versa), capturing changes in the steepness of the term structure. This component is interpreted as the **Slope Factor**.

This factor structure directly explains the maturity-dependent behavior observed in the CIR reconstruction. The single-factor CIR model is well-suited to capturing Level dynamics, but it has no mechanism to represent Slope movements. As maturity increases, slope contributions to yield variation become increasingly important, which accounts for the observed deterioration in CIR reconstruction quality at longer tenors.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(factor_df.index, factor_df["Factor1"],
        color='#FF6B6B', linewidth=2.2, label="PC1 — Level")
ax.plot(factor_df.index, factor_df["Factor2"],
        color='#4D96FF', linewidth=2.2, linestyle='-.', label="PC2 — Slope")

ax.legend()
ax.set_title("Yield Curve Factor Dynamics: Level and Slope Components")
ax.set_xlabel("Date")
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import r2_score

X = df[yield_cols].values

# Transform to PCA space
scores = pca.transform(X_scaled)

# Keep only first two PCs
scores_pc2 = scores.copy()
scores_pc2[:, 2:] = 0

# Reconstruct
X_pc2_scaled = pca.inverse_transform(scores_pc2)

X_pc2 = scaler.inverse_transform(X_pc2_scaled)

r2_pc2 = r2_score(
    X.flatten(),
    X_pc2.flatten()
)

print(f"PC1 + PC2 Reconstruction R²: {r2_pc2:.6f}")

In [ ]:
results_pc2 = []

for i, maturity in enumerate(maturities):

    r2 = r2_score(
        X[:, i],
        X_pc2[:, i]
    )

    results_pc2.append([maturity, r2])

pc2_df = pd.DataFrame(
    results_pc2,
    columns=["Maturity", "R2_PC1_PC2"]
)

pc2_df

### Two-Factor Reconstruction Validation

To quantify how much information is contained in the first two principal components, yield curves were reconstructed using only PC1 and PC2. The reconstruction achieved an overall **R² of 0.9994**, with near-perfect fit at every maturity.

This result confirms that the yield curve is almost entirely described by a **Level–Slope system**. The remaining variation unexplained by two factors is negligible, and effectively all economically meaningful yield curve dynamics are captured within this two-dimensional representation.


Further analysis reveals that the Level factor follows a non-stationary, regime-dependent trajectory, while the Slope factor exhibits cleaner mean-reverting behavior. This distinction motivates the subsequent extension, which explores a hybrid approach where the Level component is treated as persistent and the Slope component is modeled with Ornstein-Uhlenbeck dynamics.


---

# Extension: Latent Factor Recovery via PCA Mapping

The PCA analysis established that the yield curve is compactly described by two latent factors — a dominant Level Factor and a secondary Slope Factor — together explaining nearly all observed term structure variation.

This section explores a natural extension: can these latent PCA factors be inferred from the observed 3-Month yield, and can the full yield curve be recovered from that inference?

The approach proceeds as follows: polynomial regression models are trained on the training set to map the 3-Month rate to each of the two PCA factors. These factor predictions are then inverted through the PCA and scaler transformations to reconstruct yield curves on the test set.

All PCA loadings, scaling parameters, and regression coefficients were estimated exclusively on training data and applied without modification to test observations, maintaining a strictly out-of-sample evaluation.


In [ ]:
eval_cols=yield_cols

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_train = df[eval_cols]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

pca = PCA(n_components=2)

scores_train = pca.fit_transform(X_train_scaled)

pc1_train = scores_train[:,0]
pc2_train = scores_train[:,1]

print("Variance Explained:")
print(pca.explained_variance_ratio_)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

X_factor = df[['ZC025YR']]

poly = PolynomialFeatures(degree=3)

X_poly = poly.fit_transform(X_factor)

In [ ]:
pc1_model = LinearRegression()

pc1_model.fit(
    X_poly,
    pc1_train
)

pc2_model = LinearRegression()

pc2_model.fit(
    X_poly,
    pc2_train
)

### Mapping the 3-Month Yield to Latent Factors

Separate cubic polynomial regression models were fitted to predict each PCA factor from the observed 3-Month yield on the training set. The estimated factor scores are then combined and inverted through the PCA transformation to reconstruct the complete yield curve. This approach is entirely data-driven and imposes no financial structure beyond the PCA decomposition learned from the training data.


In [ ]:
X_test = test_3m[['ZC025YR']]

X_test_poly = poly.transform(X_test)

pc1_pred = pc1_model.predict(X_test_poly)

pc2_pred = pc2_model.predict(X_test_poly)

In [ ]:
import numpy as np

pred_scores = np.column_stack([
    pc1_pred,
    pc2_pred
])

reconstructed_scaled = pca.inverse_transform(
    pred_scores
)

reconstructed = scaler.inverse_transform(
    reconstructed_scaled
)

In [ ]:
pred_pca = pd.DataFrame(
    reconstructed,
    columns=eval_cols
)

pred_pca.head()

In [ ]:
from sklearn.metrics import r2_score

overall_r2 = r2_score(
    test_full[eval_cols].values.flatten(),
    pred_pca[eval_cols].values.flatten()
)

print("Overall PCA Extension R² =", overall_r2)

In [ ]:
for col in eval_cols:

    r2 = r2_score(
        test_full[col],
        pred_pca[col]
    )

    print(col, ":", round(r2,4))

### PCA Extension: Results and Analysis

The PCA-based reconstruction achieved an overall **out-of-sample R² of 0.7855**. Accuracy was strong at the short end but degraded progressively with maturity.

| Maturity | R² |
|----------|------|
| 3M | 0.9985 |
| 6M | 0.9025 |
| 9M | 0.7272 |
| 1Y | 0.4307 |
| 2Y | 0.5753 |

The results indicate that while the **Level Factor** is reasonably predictable from the 3-Month yield, the **Slope Factor** cannot be recovered with sufficient precision from the short rate alone. As maturity lengthens, unrecovered slope dynamics increasingly dominate the residual, which limits reconstruction accuracy.

Although this extension did not surpass the calibrated CIR model in overall performance, it yielded an important structural insight: the yield curve contains multi-dimensional information that cannot be fully extracted from any single observable. This finding reinforces the interpretation of CIR's maturity-dependent performance degradation and motivates exploration of richer, multi-factor specifications.


# Supervised Learning Benchmark

The CIR model imposes theoretically motivated structure on the yield curve reconstruction problem, which both constrains and guides the estimation. To assess how much predictive performance is achievable without such constraints, a selection of supervised learning algorithms was evaluated on the same train-test split.

These models learn the mapping from the 3-Month yield directly from data, without any assumptions about mean reversion, diffusion processes, or latent factor structure.

**The purpose of this benchmark is not to replace the CIR model — which retains clear economic interpretability — but to establish a data-driven performance ceiling for comparison.**


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import pandas as pd

X_train = df[['ZC025YR']]

y_train = df[
    [
        'ZC050YR',
        'ZC075YR',
        'ZC100YR',
        'ZC200YR'
    ]
]

X_test = test_3m[['ZC025YR']]

eval_cols = [
    'ZC025YR',
    'ZC050YR',
    'ZC075YR',
    'ZC100YR',
    'ZC200YR'
]

results = []

### Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso

alphas = [
    0.00001,
    0.0001,
    0.001,
    0.01
]

best_r2 = -999
best_alpha = None

for alpha in alphas:

    lasso = Lasso(alpha=alpha)

    lasso.fit(X_train,y_train)

    pred = lasso.predict(X_test)

    pred_df = pd.DataFrame(
        pred,
        columns=[
            'ZC050YR',
            'ZC075YR',
            'ZC100YR',
            'ZC200YR'
        ]
    )

    pred_df['ZC025YR'] = test_3m['ZC025YR'].values

    pred_df = pred_df[
        eval_cols
    ]

    r2 = r2_score(
        test_full[eval_cols].values.flatten(),
        pred_df[eval_cols].values.flatten()
    )

    print(
        f"alpha={alpha} -> R²={r2:.5f}"
    )

    if r2 > best_r2:

        best_r2 = r2
        best_alpha = alpha

results.append({
    "Model": "Lasso",
    "Best Parameters": f"alpha={best_alpha}",
    "R2": round(best_r2, 5)
})

### Ridge Regression

In [ ]:
alphas = [0.001,0.01,0.1,1,10,100]

best_r2 = -999
best_alpha = None

for alpha in alphas:

    ridge = Ridge(alpha=alpha)

    ridge.fit(X_train,y_train)

    pred = ridge.predict(X_test)

    pred_df = pd.DataFrame(
        pred,
        columns=[
            'ZC050YR',
            'ZC075YR',
            'ZC100YR',
            'ZC200YR'
        ]
    )

    pred_df['ZC025YR'] = test_3m['ZC025YR'].values

    pred_df = pred_df[
        eval_cols
    ]

    r2 = r2_score(
        test_full[eval_cols].values.flatten(),
        pred_df[eval_cols].values.flatten()
    )

    print(
        f"alpha={alpha} -> R²={r2:.5f}"
    )

    if r2 > best_r2:

        best_r2 = r2
        best_alpha = alpha

results.append({
    "Model": "Ridge",
    "Best Parameters": f"alpha={best_alpha}",
    "R2": round(best_r2, 5)
})

### ElasticNet

In [ ]:
from sklearn.linear_model import ElasticNet

best_r2 = -999
best_params = None

for alpha in [0.0001,0.001,0.01]:

    for l1_ratio in [0.2,0.5,0.8]:

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_test
        )

        pred_df = pd.DataFrame(
            pred,
            columns=[
                'ZC050YR',
                'ZC075YR',
                'ZC100YR',
                'ZC200YR'
            ]
        )

        pred_df['ZC025YR'] = (
            test_3m['ZC025YR'].values
        )

        pred_df = pred_df[
            eval_cols
        ]

        r2 = r2_score(
            test_full[eval_cols]
            .values.flatten(),

            pred_df[eval_cols]
            .values.flatten()
        )

        print(
            alpha,
            l1_ratio,
            r2
        )

        if r2 > best_r2:

            best_r2 = r2

            best_params = (
                alpha,
                l1_ratio
            )

results.append({
    "Model": "ElasticNet",
    "Best Parameters": str(best_params),
    "R2": round(best_r2, 5)
})

### K-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

best_r2 = -999
best_k = None

for k in [3,5,10,20,30]:

    knn = KNeighborsRegressor(
        n_neighbors=k
    )

    knn.fit(
        X_train,
        y_train
    )

    pred = knn.predict(
        X_test
    )

    pred_df = pd.DataFrame(
        pred,
        columns=[
            'ZC050YR',
            'ZC075YR',
            'ZC100YR',
            'ZC200YR'
        ]
    )

    pred_df['ZC025YR'] = (
        test_3m['ZC025YR']
    )

    pred_df = pred_df[
        eval_cols
    ]

    r2 = r2_score(
        test_full[eval_cols]
        .values.flatten(),

        pred_df[eval_cols]
        .values.flatten()
    )

    print(
        f"k={k} -> {r2:.5f}"
    )

    if r2 > best_r2:

        best_r2 = r2
        best_k = k

results.append({
    "Model": "KNN",
    "Best Parameters": f"k={best_k}",
    "R2": round(best_r2, 5)
})

### XGBoost (Gradient Boosted Trees)

In [ ]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score
import pandas as pd

best_r2 = -999
best_params = None

for n_estimators in [100,300,500]:

    for learning_rate in [0.01,0.05,0.1]:

        for max_depth in [2,3,4]:

            model = MultiOutputRegressor(

                XGBRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    random_state=42
                )
            )

            model.fit(
                X_train,
                y_train
            )

            pred = model.predict(
                X_test
            )

            pred_df = pd.DataFrame(
                pred,
                columns=[
                    'ZC050YR',
                    'ZC075YR',
                    'ZC100YR',
                    'ZC200YR'
                ]
            )

            pred_df['ZC025YR'] = (
                test_3m['ZC025YR'].values
            )

            pred_df = pred_df[
                eval_cols
            ]

            r2 = r2_score(
                test_full[eval_cols]
                .values.flatten(),

                pred_df[eval_cols]
                .values.flatten()
            )

            print(
                n_estimators,
                learning_rate,
                max_depth,
                round(r2,5)
            )

            if r2 > best_r2:

                best_r2 = r2

                best_params = (
                    n_estimators,
                    learning_rate,
                    max_depth
                )

results.append({
    "Model": "XGBoost",
    "Best Parameters": str(best_params),
    "R2": round(best_r2, 5)
})

### Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score
import pandas as pd

best_r2 = -999
best_params = None

for n_estimators in [100,300,500]:

    for learning_rate in [0.01,0.05,0.1]:

        for max_depth in [2,3]:

            model = MultiOutputRegressor(
                GradientBoostingRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    random_state=42
                )
            )

            model.fit(
                X_train,
                y_train
            )

            pred = model.predict(
                X_test
            )

            pred_df = pd.DataFrame(
                pred,
                columns=[
                    'ZC050YR',
                    'ZC075YR',
                    'ZC100YR',
                    'ZC200YR'
                ]
            )

            pred_df['ZC025YR'] = (
                test_3m['ZC025YR'].values
            )

            pred_df = pred_df[
                eval_cols
            ]

            r2 = r2_score(
                test_full[eval_cols]
                .values.flatten(),

                pred_df[eval_cols]
                .values.flatten()
            )

            print(
                n_estimators,
                learning_rate,
                max_depth,
                round(r2,5)
            )

            if r2 > best_r2:

                best_r2 = r2

                best_params = (
                    n_estimators,
                    learning_rate,
                    max_depth
                )

results.append({
    "Model": "Gradient Boosting",
    "Best Parameters": str(best_params),
    "R2": round(best_r2, 5)
})

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

results_df.index += 1

results_df.style\
    .background_gradient(subset=["R2"])\
    .format({"R2":"{:.5f}"})

results_df

### Benchmark Results: Summary

The supervised learning evaluation revealed a clear preference for linear models in this reconstruction setting.

| Rank | Model | Best R² |
|------|-------|---------|
| 1 | Ridge Regression | **0.9462** |
| 2 | ElasticNet | 0.8914 |
| 3 | Lasso | 0.8895 |
| 4 | Gradient Boosting | 0.8020 |
| 5 | XGBoost | 0.8010 |
| 6 | KNN | 0.3779 |

**Ridge Regression** achieved the highest reconstruction accuracy at **R² = 0.9462**, outperforming the calibrated CIR model (**R² ≈ 0.8945**) and the PCA extension (**R² ≈ 0.7855**).

A notable finding is that the top three models — Ridge, ElasticNet, and Lasso — are all linear, while more expressive methods (XGBoost, Gradient Boosting, KNN) performed materially worse. This pattern suggests that the statistical relationship between the 3-Month yield and longer maturities is predominantly linear over the evaluation horizon, and that additional model flexibility introduces overfitting rather than genuine predictive gain.

Ridge Regression's strong performance likely stems from its ability to exploit the high collinearity among adjacent maturities while keeping coefficient estimates stable through L2 regularization. Tree-based methods, by contrast, appear to overfit the single-feature input space.

The benchmark demonstrates that purely empirical methods can attain higher predictive scores than theoretically motivated alternatives in this setting — though at the cost of economic interpretability and the distributional guarantees provided by the CIR framework.


---

# Conclusion

This study examined yield curve reconstruction using the Cox-Ingersoll-Ross (CIR) model, with the 3-Month Treasury yield as the sole observable input. Following parameter calibration on historical training data and application of the CIR term structure pricing formulas, the model achieved an out-of-sample **R² of 0.8945**, exceeding the target benchmark.

A Principal Component Analysis of the yield curve provided deeper insight into the model's behavior. The term structure was found to be governed almost entirely by two latent components: a dominant **Level Factor** responsible for parallel shifts across the curve, and a secondary **Slope Factor** that accounts for changes in the steepness of the term structure. Together, these two factors explained nearly all observed yield variation (two-factor R² ≈ 0.9994).

An extension was developed that attempted to infer these latent factors from the 3-Month yield using polynomial regression, followed by PCA-based curve reconstruction. The approach yielded an out-of-sample **R² of 0.7855** and confirmed that while the Level Factor is largely recoverable from the short rate, the Slope Factor cannot be reliably inferred from a single maturity observation.

Finally, a machine learning benchmark demonstrated that **Ridge Regression** — a simple regularized linear model — achieved the highest out-of-sample performance at **R² = 0.9462**, outperforming both the CIR model and the PCA extension. The dominance of linear models over more complex alternatives suggests that the yield curve reconstruction problem, in this dataset, is fundamentally linear in nature.

The key takeaway is that calibration methodology and the degree to which the model's assumptions align with the data's factor structure are the primary determinants of reconstruction quality. The CIR model offers interpretability and economic grounding that purely data-driven methods cannot provide, making it a valuable tool even when its predictive accuracy is not maximal.


## Directions for Future Research

Several extensions could improve upon the results presented here. Multi-factor affine term structure models — such as the two-factor CIR or Vasicek variants — would allow explicit modeling of the Slope Factor identified by PCA, likely improving long-maturity reconstruction. Regime-switching specifications could better accommodate the structural breaks visible in the historical rate data. Kalman Filter-based state-space estimation offers a principled way to jointly estimate latent factors and model parameters in a unified framework, potentially combining the interpretability of CIR with the flexibility needed to track non-stationary dynamics.
